In [ ]:
import psycopg2
import pandas as pd
import numpy as np

from IPython.display import Javascript, display
import ipywidgets as widgets
from ipywidgets import interact, Dropdown,Button

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from rdkit.Chem import PandasTools as PandasTools

from molvs import Standardizer
from molvs.errors import MolVSError

from pandarallel import pandarallel as para

import sklearn

# Java
from py4j.java_gateway import JavaGateway
from IPython.display import Javascript, display


## 1. Data upload and standardization

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)  # or 1000consen
pd.set_option('display.max_columns', None)  # or 1000

In [ ]:
df = pd.read_csv('your_data.csv') #enter the name of your input file

### Standardization

In [ ]:
# Displays the version used in this JN
print(" RDKit Version " + rdkit.__version__ + " is used.")
print(" Pandas Version " + pd.__version__ + " is used.")
pd.set_option('display.max_colwidth', None)
PandasTools.RenderImagesInAllDataFrames(images=True)
para.initialize(progress_bar=True)

In [ ]:
import pandas as pd
from rdkit import Chem
from molvs import standardize_smiles

# checks if the retrieved compound has non-organic fragments
def contains_nonorg(fragment):
    # organic: H, C, N, O, P, S, F, Cl, Br, I
    for a in fragment.GetAtoms():
        if a.GetAtomicNum() not in [1, 6, 7, 8, 15, 16, 9, 17, 35, 53]:
            return "Yes"
    return "No"

##########################################################################################
#inserted by me:
def is_nonorganic(mol):
    # Replace this with your logic.
    # Here's an example that returns True if no carbon atoms are present:
    if mol is None:
        return True
    try:
        atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
        return "C" not in atoms
    except:
        return True
###########################################################################################

# standardisation, fragment removal, uncharge molecules, remove stereochemistry
def standardise_remove_fragments_uncharge_removesteochemistry(df):
    # adds the molvs standardizer as a variable
    s = Standardizer()
    # checks if contains non organic fragment
    fragment_nonorganic = df['ROMol'].parallel_apply(lambda x:contains_nonorg(x))
    df = df.assign(fragment_nonorganic=fragment_nonorganic)
    # standardizes each compound
    ROMol_stand = df['ROMol'].parallel_apply(lambda x:s.standardize(x))
    df = df.assign(ROMol_stand=ROMol_stand)    
    # removes fragments from each compound using molvs
    ROMol_frag = df['ROMol_stand'].parallel_apply(lambda x:s.remove_fragments(x))
    df = df.assign(ROMol_frag=ROMol_frag)
    # keeps the larger fragment in each compound 
    ROMol_frag2 = df['ROMol_stand'].parallel_apply(lambda x:s.largest_fragment(x))
    df = df.assign(ROMol_frag2=ROMol_frag2)
    # removes stereochemistry in each compound
    ROMol_stereo = df['ROMol_frag2'].parallel_apply(lambda x:s.stereo_parent(x))
    df = df.assign(ROMol_stereo=ROMol_stereo)
    # removes the charges from each compounds    
    ROMol_charge = df['ROMol_stereo'].parallel_apply(lambda x:s.uncharge(x))
    df = df.assign(ROMol_charge=ROMol_charge)
    # checks if molecule is organic or inorganic
    inorganic = df['ROMol_charge'].parallel_apply(lambda x:is_nonorganic(x))
    df = df.assign(inorganic=inorganic)
    # adds a column with the final ROMol formate
    df = df.assign(ROMol_fin=ROMol_charge)
    return(df)

# calculates the InChIs, SMILES, InChIKeys
def calc_InCHI_SMILES_InChIKey(df):
    # calculates InChIs for each standarized compound
    df['InChIs_stand'] = df['ROMol_fin'].parallel_apply(lambda x:AllChem.MolToInchi(x))
    # calculates SMILES for each standarized compound
    df['SMILES_stand'] = df['ROMol_fin'].parallel_apply(lambda x:AllChem.MolToSmiles(x))
    # calculates InChIkeys for each compound
    df['InChIKey_stand'] = df['ROMol_fin'].parallel_apply(lambda x:AllChem.MolToInchiKey(x))
    return(df)

# removes stereochemistry information
def removeStereoInfo_from_InCHhI(fullInchi, position, delimiter):
    return delimiter.join(fullInchi.split(delimiter)[:position])

# used to identify molerror during standardisation
def molerror(mol):
    # gives an error if a problem occurs
    err = MolVSError()
    try:
        return err.with_traceback(mol)
    except:
        pass

In [ ]:
# Creates the ROMol Format and correspodning column

ROMol = df['Smiles'].astype(str).map(lambda x:AllChem.MolFromSmiles(x))
df_ROMol = df.assign(ROMol=ROMol)

In [ ]:
# Drop nans in column 'ROMol' since causes problems when creating an SDF-file later on
df_cleaned = df_ROMol.dropna(subset=['ROMol'])

In [ ]:
initial_count = len(df_ROMol)
df_cleaned = df_ROMol.dropna(subset=['ROMol'])
final_count = len(df_cleaned)

dropped = initial_count - final_count
print(f"Rows dropped: {dropped}")


#### SDF generation for docking (before standardization because we want to use ligprep)

In [ ]:
from rdkit.Chem import PandasTools

PandasTools.AddMoleculeColumnToFrame(df_cleaned, smilesCol='Smiles', molCol='ROMol', includeFingerprints=False)

PandasTools.WriteSDF(df_cleaned, 'file.sdf', molColName='ROMol', properties=list(df_cleaned.columns.drop('ROMol')))


#### Standardization for ML

In [ ]:
# Compounds are standarised
molecules_stand = standardise_remove_fragments_uncharge_removesteochemistry(df_cleaned)

#### Calculation of InChIs, SMILES and InChIKeys

In [ ]:
# Calculation of InChIs, SMILES and InChIKeys
molecules_stand = calc_InCHI_SMILES_InChIKey(molecules_stand)

#### Removal of Steroechemistry in InChI for Duplicate Check

In [ ]:
# Removal of stereochemistry from InChIs
molecules_stand['InChI_steorem'] = molecules_stand['InChIs_stand'].map(lambda x:removeStereoInfo_from_InCHhI(x,4,"/"))

#### Standardisation Check

In [ ]:
# Standardisation Check
molecules_stand['molerror'] = pd.DataFrame(molecules_stand['ROMol_fin'].parallel_apply(lambda x:molerror(x)))
molecules_error = molecules_stand[~molecules_stand['molerror'].isnull()]
molecules_non_error = molecules_stand[molecules_stand['molerror'].isnull()]
print (str(len(molecules_non_error)) + ' of ' + str(len(molecules_stand)) + ' compounds from the data set could be standardised.')

#### Removal of Duplicates

In [ ]:
# Removal of duplicates
molecules_stand_uniq = molecules_stand.drop_duplicates(subset='InChI_steorem',keep='first')
molecules_stand_dupl = molecules_stand[molecules_stand.duplicated(['InChI_steorem'], keep='first')]

#### Change Column Name ROMol_fin to ROMol and ROMol to ROMOl_init 

In [ ]:
# Column name change
SDF = molecules_stand_uniq.rename(columns = {'ROMol':'ROMol_init', 'ROMol_fin':'ROMol'}, inplace = True)

In [ ]:
# Drops stepwise standardized ROMol formats
SDF = molecules_stand_uniq.drop(['ROMol_init','ROMol_stand','ROMol_frag','ROMol_frag2', 'ROMol_stereo','ROMol_charge'], axis=1)

#### Drop columns with inorganic compounds

In [ ]:
# Initial number of compounds
initial_count = len(SDF)

# Filter out inorganic compounds
SDF_fin = SDF[SDF['inorganic'] == False]

# Final number of compounds (after filtering)
final_count = len(SDF_fin)

# Calculate the number of dropped compounds
dropped_count = initial_count - final_count

# Print the number of dropped compounds
print(f"Number of compounds dropped: {dropped_count}")

In [ ]:
# Checks if compounds are inorganic
SDF_fin = SDF[SDF['inorganic'] == False]

In [ ]:
SDF_fin.to_csv('file_standardized.csv', index=False)

## 2. Check which compounds are in training set

In [ ]:
df_training_set = pd.read_csv('NIS_stand_CDDD_filtered.csv')

In [ ]:
test_inchis = set(SDF_fin['InChIs_stand'].dropna())
train_inchis = set(df_training_set['InChIs_stand'].dropna())

overlaps = test_inchis.intersection(train_inchis)

print(f"Test set size: {len(test_inchis)} unique InChIs")
print(f"Training set size: {len(train_inchis)} unique InChIs")
print(f"Overlapping InChIs: {len(overlaps)}")
print(f"Overlap percentage: {len(overlaps)/len(test_inchis)*100:.2f}% of test set")

In [ ]:
# Create overlap table with specific columns from training set
overlap_table = df_training_set[
    df_training_set['InChIs_stand'].isin(overlaps)
][['Substance_Name', 'InChIs_stand', 'NIS Hit2']].drop_duplicates()

# Merge with SDF_fin to bring in 'Preferred Name'
overlap_table = overlap_table.merge(
    SDF_fin[['InChIs_stand', 'Preferred Name']],
    on='InChIs_stand',
    how='left'
)

print(f"Overlap table with {len(overlap_table)} compounds")


overlap_table.to_csv('training_set_overlaps.csv', index=False)
print(f"\nSaved overlap table to 'training_set_overlaps.csv'")

In [ ]:
# Create clean test set without overlaps
SDF_fin_clean = SDF_fin[~SDF_fin['InChIs_stand'].isin(overlaps)].copy()

print(f"Original test set (SDF_fin): {len(SDF_fin)} compounds")
print(f"Clean test set: {len(SDF_fin_clean)} compounds")
print(f"Excluded: {len(SDF_fin) - len(SDF_fin_clean)} overlapping compounds")

# Save clean test set
SDF_fin_clean.to_csv('SDF_fin_clean.csv', index=False)
print(f"Saved clean test set to 'SDF_fin_clean.csv'")


## 3. CDDD generation

In [ ]:
#https://github.com/jrwnter/cddd
#use SDF_fin_clean.csv for CDDD generation

In [ ]:
#after CDDD generation put file into folder "test_set"

## 4. ML

In [ ]:
import pandas as pd
from string import Template
import json
import numpy as np
from itertools import islice

#import seaborn as sns
import matplotlib.pyplot as plt

import os
import glob

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
#from xgboost import XGBClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, balanced_accuracy_score, matthews_corrcoef
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer
from collections import Counter
import joblib
import dill

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors
from rdkit.Chem import PandasTools as PandasTools
from rdkit.Chem.PandasTools import LoadSDF

from concurrent.futures import ProcessPoolExecutor

# Java
from py4j.java_gateway import JavaGateway
from IPython.display import Javascript, display

In [ ]:
def preprocess_data(df, descriptor_set):
    df_filtered = df[descriptor_set].dropna()
    X = df_filtered.values
    return X

In [ ]:
ls_descs = [ 'CDDD fingerprint[1]',
 'CDDD fingerprint[2]',
 'CDDD fingerprint[3]',
 'CDDD fingerprint[4]',
 'CDDD fingerprint[5]',
 'CDDD fingerprint[6]',
 'CDDD fingerprint[7]',
 'CDDD fingerprint[8]',
 'CDDD fingerprint[9]',
 'CDDD fingerprint[10]',
 'CDDD fingerprint[11]',
 'CDDD fingerprint[12]',
 'CDDD fingerprint[13]',
 'CDDD fingerprint[14]',
 'CDDD fingerprint[15]',
 'CDDD fingerprint[16]',
 'CDDD fingerprint[17]',
 'CDDD fingerprint[18]',
 'CDDD fingerprint[19]',
 'CDDD fingerprint[20]',
 'CDDD fingerprint[21]',
 'CDDD fingerprint[22]',
 'CDDD fingerprint[23]',
 'CDDD fingerprint[24]',
 'CDDD fingerprint[25]',
 'CDDD fingerprint[26]',
 'CDDD fingerprint[27]',
 'CDDD fingerprint[28]',
 'CDDD fingerprint[29]',
 'CDDD fingerprint[30]',
 'CDDD fingerprint[31]',
 'CDDD fingerprint[32]',
 'CDDD fingerprint[33]',
 'CDDD fingerprint[34]',
 'CDDD fingerprint[35]',
 'CDDD fingerprint[36]',
 'CDDD fingerprint[37]',
 'CDDD fingerprint[38]',
 'CDDD fingerprint[39]',
 'CDDD fingerprint[40]',
 'CDDD fingerprint[41]',
 'CDDD fingerprint[42]',
 'CDDD fingerprint[43]',
 'CDDD fingerprint[44]',
 'CDDD fingerprint[45]',
 'CDDD fingerprint[46]',
 'CDDD fingerprint[47]',
 'CDDD fingerprint[48]',
 'CDDD fingerprint[49]',
 'CDDD fingerprint[50]',
 'CDDD fingerprint[51]',
 'CDDD fingerprint[52]',
 'CDDD fingerprint[53]',
 'CDDD fingerprint[54]',
 'CDDD fingerprint[55]',
 'CDDD fingerprint[56]',
 'CDDD fingerprint[57]',
 'CDDD fingerprint[58]',
 'CDDD fingerprint[59]',
 'CDDD fingerprint[60]',
 'CDDD fingerprint[61]',
 'CDDD fingerprint[62]',
 'CDDD fingerprint[63]',
 'CDDD fingerprint[64]',
 'CDDD fingerprint[65]',
 'CDDD fingerprint[66]',
 'CDDD fingerprint[67]',
 'CDDD fingerprint[68]',
 'CDDD fingerprint[69]',
 'CDDD fingerprint[70]',
 'CDDD fingerprint[71]',
 'CDDD fingerprint[72]',
 'CDDD fingerprint[73]',
 'CDDD fingerprint[74]',
 'CDDD fingerprint[75]',
 'CDDD fingerprint[76]',
 'CDDD fingerprint[77]',
 'CDDD fingerprint[78]',
 'CDDD fingerprint[79]',
 'CDDD fingerprint[80]',
 'CDDD fingerprint[81]',
 'CDDD fingerprint[82]',
 'CDDD fingerprint[83]',
 'CDDD fingerprint[84]',
 'CDDD fingerprint[85]',
 'CDDD fingerprint[86]',
 'CDDD fingerprint[87]',
 'CDDD fingerprint[88]',
 'CDDD fingerprint[89]',
 'CDDD fingerprint[90]',
 'CDDD fingerprint[91]',
 'CDDD fingerprint[92]',
 'CDDD fingerprint[93]',
 'CDDD fingerprint[94]',
 'CDDD fingerprint[95]',
 'CDDD fingerprint[96]',
 'CDDD fingerprint[97]',
 'CDDD fingerprint[98]',
 'CDDD fingerprint[99]',
 'CDDD fingerprint[100]',
 'CDDD fingerprint[101]',
 'CDDD fingerprint[102]',
 'CDDD fingerprint[103]',
 'CDDD fingerprint[104]',
 'CDDD fingerprint[105]',
 'CDDD fingerprint[106]',
 'CDDD fingerprint[107]',
 'CDDD fingerprint[108]',
 'CDDD fingerprint[109]',
 'CDDD fingerprint[110]',
 'CDDD fingerprint[111]',
 'CDDD fingerprint[112]',
 'CDDD fingerprint[113]',
 'CDDD fingerprint[114]',
 'CDDD fingerprint[115]',
 'CDDD fingerprint[116]',
 'CDDD fingerprint[117]',
 'CDDD fingerprint[118]',
 'CDDD fingerprint[119]',
 'CDDD fingerprint[120]',
 'CDDD fingerprint[121]',
 'CDDD fingerprint[122]',
 'CDDD fingerprint[123]',
 'CDDD fingerprint[124]',
 'CDDD fingerprint[125]',
 'CDDD fingerprint[126]',
 'CDDD fingerprint[127]',
 'CDDD fingerprint[128]',
 'CDDD fingerprint[129]',
 'CDDD fingerprint[130]',
 'CDDD fingerprint[131]',
 'CDDD fingerprint[132]',
 'CDDD fingerprint[133]',
 'CDDD fingerprint[134]',
 'CDDD fingerprint[135]',
 'CDDD fingerprint[136]',
 'CDDD fingerprint[137]',
 'CDDD fingerprint[138]',
 'CDDD fingerprint[139]',
 'CDDD fingerprint[140]',
 'CDDD fingerprint[141]',
 'CDDD fingerprint[142]',
 'CDDD fingerprint[143]',
 'CDDD fingerprint[144]',
 'CDDD fingerprint[145]',
 'CDDD fingerprint[146]',
 'CDDD fingerprint[147]',
 'CDDD fingerprint[148]',
 'CDDD fingerprint[149]',
 'CDDD fingerprint[150]',
 'CDDD fingerprint[151]',
 'CDDD fingerprint[152]',
 'CDDD fingerprint[153]',
 'CDDD fingerprint[154]',
 'CDDD fingerprint[155]',
 'CDDD fingerprint[156]',
 'CDDD fingerprint[157]',
 'CDDD fingerprint[158]',
 'CDDD fingerprint[159]',
 'CDDD fingerprint[160]',
 'CDDD fingerprint[161]',
 'CDDD fingerprint[162]',
 'CDDD fingerprint[163]',
 'CDDD fingerprint[164]',
 'CDDD fingerprint[165]',
 'CDDD fingerprint[166]',
 'CDDD fingerprint[167]',
 'CDDD fingerprint[168]',
 'CDDD fingerprint[169]',
 'CDDD fingerprint[170]',
 'CDDD fingerprint[171]',
 'CDDD fingerprint[172]',
 'CDDD fingerprint[173]',
 'CDDD fingerprint[174]',
 'CDDD fingerprint[175]',
 'CDDD fingerprint[176]',
 'CDDD fingerprint[177]',
 'CDDD fingerprint[178]',
 'CDDD fingerprint[179]',
 'CDDD fingerprint[180]',
 'CDDD fingerprint[181]',
 'CDDD fingerprint[182]',
 'CDDD fingerprint[183]',
 'CDDD fingerprint[184]',
 'CDDD fingerprint[185]',
 'CDDD fingerprint[186]',
 'CDDD fingerprint[187]',
 'CDDD fingerprint[188]',
 'CDDD fingerprint[189]',
 'CDDD fingerprint[190]',
 'CDDD fingerprint[191]',
 'CDDD fingerprint[192]',
 'CDDD fingerprint[193]',
 'CDDD fingerprint[194]',
 'CDDD fingerprint[195]',
 'CDDD fingerprint[196]',
 'CDDD fingerprint[197]',
 'CDDD fingerprint[198]',
 'CDDD fingerprint[199]',
 'CDDD fingerprint[200]',
 'CDDD fingerprint[201]',
 'CDDD fingerprint[202]',
 'CDDD fingerprint[203]',
 'CDDD fingerprint[204]',
 'CDDD fingerprint[205]',
 'CDDD fingerprint[206]',
 'CDDD fingerprint[207]',
 'CDDD fingerprint[208]',
 'CDDD fingerprint[209]',
 'CDDD fingerprint[210]',
 'CDDD fingerprint[211]',
 'CDDD fingerprint[212]',
 'CDDD fingerprint[213]',
 'CDDD fingerprint[214]',
 'CDDD fingerprint[215]',
 'CDDD fingerprint[216]',
 'CDDD fingerprint[217]',
 'CDDD fingerprint[218]',
 'CDDD fingerprint[219]',
 'CDDD fingerprint[220]',
 'CDDD fingerprint[221]',
 'CDDD fingerprint[222]',
 'CDDD fingerprint[223]',
 'CDDD fingerprint[224]',
 'CDDD fingerprint[225]',
 'CDDD fingerprint[226]',
 'CDDD fingerprint[227]',
 'CDDD fingerprint[228]',
 'CDDD fingerprint[229]',
 'CDDD fingerprint[230]',
 'CDDD fingerprint[231]',
 'CDDD fingerprint[232]',
 'CDDD fingerprint[233]',
 'CDDD fingerprint[234]',
 'CDDD fingerprint[235]',
 'CDDD fingerprint[236]',
 'CDDD fingerprint[237]',
 'CDDD fingerprint[238]',
 'CDDD fingerprint[239]',
 'CDDD fingerprint[240]',
 'CDDD fingerprint[241]',
 'CDDD fingerprint[242]',
 'CDDD fingerprint[243]',
 'CDDD fingerprint[244]',
 'CDDD fingerprint[245]',
 'CDDD fingerprint[246]',
 'CDDD fingerprint[247]',
 'CDDD fingerprint[248]',
 'CDDD fingerprint[249]',
 'CDDD fingerprint[250]',
 'CDDD fingerprint[251]',
 'CDDD fingerprint[252]',
 'CDDD fingerprint[253]',
 'CDDD fingerprint[254]',
 'CDDD fingerprint[255]',
 'CDDD fingerprint[256]',
 'CDDD fingerprint[257]',
 'CDDD fingerprint[258]',
 'CDDD fingerprint[259]',
 'CDDD fingerprint[260]',
 'CDDD fingerprint[261]',
 'CDDD fingerprint[262]',
 'CDDD fingerprint[263]',
 'CDDD fingerprint[264]',
 'CDDD fingerprint[265]',
 'CDDD fingerprint[266]',
 'CDDD fingerprint[267]',
 'CDDD fingerprint[268]',
 'CDDD fingerprint[269]',
 'CDDD fingerprint[270]',
 'CDDD fingerprint[271]',
 'CDDD fingerprint[272]',
 'CDDD fingerprint[273]',
 'CDDD fingerprint[274]',
 'CDDD fingerprint[275]',
 'CDDD fingerprint[276]',
 'CDDD fingerprint[277]',
 'CDDD fingerprint[278]',
 'CDDD fingerprint[279]',
 'CDDD fingerprint[280]',
 'CDDD fingerprint[281]',
 'CDDD fingerprint[282]',
 'CDDD fingerprint[283]',
 'CDDD fingerprint[284]',
 'CDDD fingerprint[285]',
 'CDDD fingerprint[286]',
 'CDDD fingerprint[287]',
 'CDDD fingerprint[288]',
 'CDDD fingerprint[289]',
 'CDDD fingerprint[290]',
 'CDDD fingerprint[291]',
 'CDDD fingerprint[292]',
 'CDDD fingerprint[293]',
 'CDDD fingerprint[294]',
 'CDDD fingerprint[295]',
 'CDDD fingerprint[296]',
 'CDDD fingerprint[297]',
 'CDDD fingerprint[298]',
 'CDDD fingerprint[299]',
 'CDDD fingerprint[300]',
 'CDDD fingerprint[301]',
 'CDDD fingerprint[302]',
 'CDDD fingerprint[303]',
 'CDDD fingerprint[304]',
 'CDDD fingerprint[305]',
 'CDDD fingerprint[306]',
 'CDDD fingerprint[307]',
 'CDDD fingerprint[308]',
 'CDDD fingerprint[309]',
 'CDDD fingerprint[310]',
 'CDDD fingerprint[311]',
 'CDDD fingerprint[312]',
 'CDDD fingerprint[313]',
 'CDDD fingerprint[314]',
 'CDDD fingerprint[315]',
 'CDDD fingerprint[316]',
 'CDDD fingerprint[317]',
 'CDDD fingerprint[318]',
 'CDDD fingerprint[319]',
 'CDDD fingerprint[320]',
 'CDDD fingerprint[321]',
 'CDDD fingerprint[322]',
 'CDDD fingerprint[323]',
 'CDDD fingerprint[324]',
 'CDDD fingerprint[325]',
 'CDDD fingerprint[326]',
 'CDDD fingerprint[327]',
 'CDDD fingerprint[328]',
 'CDDD fingerprint[329]',
 'CDDD fingerprint[330]',
 'CDDD fingerprint[331]',
 'CDDD fingerprint[332]',
 'CDDD fingerprint[333]',
 'CDDD fingerprint[334]',
 'CDDD fingerprint[335]',
 'CDDD fingerprint[336]',
 'CDDD fingerprint[337]',
 'CDDD fingerprint[338]',
 'CDDD fingerprint[339]',
 'CDDD fingerprint[340]',
 'CDDD fingerprint[341]',
 'CDDD fingerprint[342]',
 'CDDD fingerprint[343]',
 'CDDD fingerprint[344]',
 'CDDD fingerprint[345]',
 'CDDD fingerprint[346]',
 'CDDD fingerprint[347]',
 'CDDD fingerprint[348]',
 'CDDD fingerprint[349]',
 'CDDD fingerprint[350]',
 'CDDD fingerprint[351]',
 'CDDD fingerprint[352]',
 'CDDD fingerprint[353]',
 'CDDD fingerprint[354]',
 'CDDD fingerprint[355]',
 'CDDD fingerprint[356]',
 'CDDD fingerprint[357]',
 'CDDD fingerprint[358]',
 'CDDD fingerprint[359]',
 'CDDD fingerprint[360]',
 'CDDD fingerprint[361]',
 'CDDD fingerprint[362]',
 'CDDD fingerprint[363]',
 'CDDD fingerprint[364]',
 'CDDD fingerprint[365]',
 'CDDD fingerprint[366]',
 'CDDD fingerprint[367]',
 'CDDD fingerprint[368]',
 'CDDD fingerprint[369]',
 'CDDD fingerprint[370]',
 'CDDD fingerprint[371]',
 'CDDD fingerprint[372]',
 'CDDD fingerprint[373]',
 'CDDD fingerprint[374]',
 'CDDD fingerprint[375]',
 'CDDD fingerprint[376]',
 'CDDD fingerprint[377]',
 'CDDD fingerprint[378]',
 'CDDD fingerprint[379]',
 'CDDD fingerprint[380]',
 'CDDD fingerprint[381]',
 'CDDD fingerprint[382]',
 'CDDD fingerprint[383]',
 'CDDD fingerprint[384]',
 'CDDD fingerprint[385]',
 'CDDD fingerprint[386]',
 'CDDD fingerprint[387]',
 'CDDD fingerprint[388]',
 'CDDD fingerprint[389]',
 'CDDD fingerprint[390]',
 'CDDD fingerprint[391]',
 'CDDD fingerprint[392]',
 'CDDD fingerprint[393]',
 'CDDD fingerprint[394]',
 'CDDD fingerprint[395]',
 'CDDD fingerprint[396]',
 'CDDD fingerprint[397]',
 'CDDD fingerprint[398]',
 'CDDD fingerprint[399]',
 'CDDD fingerprint[400]',
 'CDDD fingerprint[401]',
 'CDDD fingerprint[402]',
 'CDDD fingerprint[403]',
 'CDDD fingerprint[404]',
 'CDDD fingerprint[405]',
 'CDDD fingerprint[406]',
 'CDDD fingerprint[407]',
 'CDDD fingerprint[408]',
 'CDDD fingerprint[409]',
 'CDDD fingerprint[410]',
 'CDDD fingerprint[411]',
 'CDDD fingerprint[412]',
 'CDDD fingerprint[413]',
 'CDDD fingerprint[414]',
 'CDDD fingerprint[415]',
 'CDDD fingerprint[416]',
 'CDDD fingerprint[417]',
 'CDDD fingerprint[418]',
 'CDDD fingerprint[419]',
 'CDDD fingerprint[420]',
 'CDDD fingerprint[421]',
 'CDDD fingerprint[422]',
 'CDDD fingerprint[423]',
 'CDDD fingerprint[424]',
 'CDDD fingerprint[425]',
 'CDDD fingerprint[426]',
 'CDDD fingerprint[427]',
 'CDDD fingerprint[428]',
 'CDDD fingerprint[429]',
 'CDDD fingerprint[430]',
 'CDDD fingerprint[431]',
 'CDDD fingerprint[432]',
 'CDDD fingerprint[433]',
 'CDDD fingerprint[434]',
 'CDDD fingerprint[435]',
 'CDDD fingerprint[436]',
 'CDDD fingerprint[437]',
 'CDDD fingerprint[438]',
 'CDDD fingerprint[439]',
 'CDDD fingerprint[440]',
 'CDDD fingerprint[441]',
 'CDDD fingerprint[442]',
 'CDDD fingerprint[443]',
 'CDDD fingerprint[444]',
 'CDDD fingerprint[445]',
 'CDDD fingerprint[446]',
 'CDDD fingerprint[447]',
 'CDDD fingerprint[448]',
 'CDDD fingerprint[449]',
 'CDDD fingerprint[450]',
 'CDDD fingerprint[451]',
 'CDDD fingerprint[452]',
 'CDDD fingerprint[453]',
 'CDDD fingerprint[454]',
 'CDDD fingerprint[455]',
 'CDDD fingerprint[456]',
 'CDDD fingerprint[457]',
 'CDDD fingerprint[458]',
 'CDDD fingerprint[459]',
 'CDDD fingerprint[460]',
 'CDDD fingerprint[461]',
 'CDDD fingerprint[462]',
 'CDDD fingerprint[463]',
 'CDDD fingerprint[464]',
 'CDDD fingerprint[465]',
 'CDDD fingerprint[466]',
 'CDDD fingerprint[467]',
 'CDDD fingerprint[468]',
 'CDDD fingerprint[469]',
 'CDDD fingerprint[470]',
 'CDDD fingerprint[471]',
 'CDDD fingerprint[472]',
 'CDDD fingerprint[473]',
 'CDDD fingerprint[474]',
 'CDDD fingerprint[475]',
 'CDDD fingerprint[476]',
 'CDDD fingerprint[477]',
 'CDDD fingerprint[478]',
 'CDDD fingerprint[479]',
 'CDDD fingerprint[480]',
 'CDDD fingerprint[481]',
 'CDDD fingerprint[482]',
 'CDDD fingerprint[483]',
 'CDDD fingerprint[484]',
 'CDDD fingerprint[485]',
 'CDDD fingerprint[486]',
 'CDDD fingerprint[487]',
 'CDDD fingerprint[488]',
 'CDDD fingerprint[489]',
 'CDDD fingerprint[490]',
 'CDDD fingerprint[491]',
 'CDDD fingerprint[492]',
 'CDDD fingerprint[493]',
 'CDDD fingerprint[494]',
 'CDDD fingerprint[495]',
 'CDDD fingerprint[496]',
 'CDDD fingerprint[497]',
 'CDDD fingerprint[498]',
 'CDDD fingerprint[499]',
 'CDDD fingerprint[500]',
 'CDDD fingerprint[501]',
 'CDDD fingerprint[502]',
 'CDDD fingerprint[503]',
 'CDDD fingerprint[504]',
 'CDDD fingerprint[505]',
 'CDDD fingerprint[506]',
 'CDDD fingerprint[507]',
 'CDDD fingerprint[508]',
 'CDDD fingerprint[509]',
 'CDDD fingerprint[510]',
 'CDDD fingerprint[511]',
 'CDDD fingerprint[512]']

In [ ]:
import joblib

# Setup base folders and paths
base_data_folder = 'test_set'  # Contains your test set CSV (output from CDDD generation)
base_model_base_folder = 'test+train_CDDD_RF_final_models_mostcommonparams_9splits'  # Base model folder that contains 9 subfolders for each split
results_folder = 'test_results'  # Where you want to save your results
descriptor_sets = ls_descs  # Define the descriptor_sets list based on your CSV columns (features)

# Locate the test set file
test_set_file = next((f for f in os.listdir(base_data_folder) if f.endswith('.csv')), None)

if test_set_file:
    # Load the test set
    test_set_path = os.path.join(base_data_folder, test_set_file)
    test_set = pd.read_csv(test_set_path)
    
    predictions_df = test_set[['Preferred Name']].copy() #enter column name with the compound identifier

    # Preprocess the test set
    X = preprocess_data(test_set, ls_descs)


    # Ensure the results folder exists
    os.makedirs(results_folder, exist_ok=True)

    # Dictionary of model filenames
    model_files = {
        'random_forest': "random_forest_final_model_split_.pkl"}

    # Loop through each split folder and each model type
    for split_num in range(1, 10):  # For splits 1 through 9
        model_folder = os.path.join(base_model_base_folder, f"Split{split_num}")
        
        # Loop through each model in the current split folder
        for model_name, filename in model_files.items():
            model_path = os.path.join(model_folder, filename)

            # Debugging: Check if model path exists
            print(f"Checking for model file: {model_path}")
            if os.path.exists(model_path):
                print(f"Loading model: {model_name} from {model_path}")
                model = joblib.load(model_path)
                # Generate predictions and add them to the DataFrame
                predictions = model.predict(X)
                predictions_df[f"{model_name}_split{split_num}_Prediction"] = predictions
                print(f"Added predictions for {model_name} in split {split_num}")
            else:
                print(f"Model file not found: {model_path}")

    # Debugging: Check final DataFrame columns
    print("Final DataFrame columns:", predictions_df.columns.tolist())

    # Save the final consolidated DataFrame to a CSV file
    consolidated_predictions_filename_csv = os.path.join(results_folder, 'consolidated_predictions.csv')
    predictions_df.to_csv(consolidated_predictions_filename_csv, index=False)


In [ ]:
# Model types to calculate sums for
model_types = ['random_forest']

# Calculate and add sum columns for rf, svm, and xgboost across all splits
for model_name in model_types:
    # Identify columns for each model type across splits
    model_prediction_columns = [f"{model_name}_split{split_num}_Prediction" for split_num in range(1, 10)]
    
    # Check that these columns exist in the DataFrame
    model_prediction_columns = [col for col in model_prediction_columns if col in predictions_df.columns]
    
    # Calculate the sum of predictions across these columns
    predictions_df[f"{model_name}_Total_Prediction"] = predictions_df[model_prediction_columns].sum(axis=1)

# Save the updated DataFrame to a new CSV file
consolidated_predictions_with_sums_filename_csv = os.path.join(results_folder, 'consolidated_predictions_with_sums.csv')
predictions_df.to_csv(consolidated_predictions_with_sums_filename_csv, index=False)


In [ ]:
# Calculate majority class vote based on the sums
# Threshold: <5 becomes 0, ≥5 becomes 1
predictions_df['random_forest_Majority_Vote'] = predictions_df['random_forest_Total_Prediction'].apply(lambda x: 1 if x > 4 else 0)

# Save the updated DataFrame to include the majority vote columns
consolidated_predictions_with_votes_filename_csv = os.path.join(results_folder, 'consolidated_predictions_with_sums_and_votes.csv')
predictions_df.to_csv(consolidated_predictions_with_votes_filename_csv, index=False)


## 5. Consensus scoring (ML + docking)

In [ ]:
df_ML = pd.read_csv('test_results/consolidated_predictions_with_sums_and_votes.csv')
df_dock = pd.read_csv('your_data_docked.csv') #enter name of your docking output files (only first ranked compounds/ one docking score per compound)
df_ligprep = pd.read_csv('your_data_ligprepped.csv') #enter name of your ligprep output file 

In [ ]:
merged_dock=pd.merge(df_ligprep, df_dock, left_on='Preferred Name', right_on='Preferred Name', how='outer') #merge ligprep output with docking output, we want to keep the compounds that were not successfully docked

In [ ]:
merged_dock['docking score'] = merged_dock['docking score'].fillna(2.8) #assign docking score of 2.8 to compounds that were not docked (it is slightly worse than the score of the worst performing docked compound from the test set)

In [ ]:
merged_df=pd.merge(df_ML, merged_dock, left_on='Preferred Name', right_on='Preferred Name', how='outer') #merge ML- and docking output

### minmax with training set borders

In [ ]:
#set predifined borders to which normalization should take place
docking_min = -12.073
docking_max = 2.8

ml_min = 0
ml_max = 9


# Invert docking score
inverted_docking_score = merged_df['docking score'] * -1

In [ ]:
#rf

# Manual normalization
merged_df['normalized_docking_score_man'] = (
    (inverted_docking_score - (-docking_max)) / ((-docking_min) - (-docking_max))
).clip(0, 1)

merged_df['normalized_random_forest_Total_Prediction_man'] = (
    (merged_df['random_forest_Total_Prediction'] - ml_min) /
    (ml_max - ml_min)
).clip(0, 1)

# Create combined score column
merged_df['normalized_sum_rf_man'] = (
    merged_df['normalized_docking_score_man'] +
    merged_df['normalized_random_forest_Total_Prediction_man']
)

In [ ]:
merged_df['predicted_activity'] = np.where(
    merged_df['normalized_sum_rf_man'].isna(),
    np.nan,
    (merged_df['normalized_sum_rf_man'] > 0.8215).astype(int) #set predifined classification threshold
)


In [ ]:
merged_df.to_csv('consensus_pred.csv', index=False)


#### Merge with initial table

In [ ]:
df_initial = pd.read_csv('your_data.csv') #your dataset that you uploaded in 1. Data upload and standardization
df_ts_overlaps = pd.read_csv('training_set_overlaps.csv') #overlaps with training set that you created in 2.

In [ ]:
pred_and_init = pd.merge(merged_df, df_initial, left_on='Preferred Name', right_on='Preferred Name', how='right') #merge predictions with initial training set

In [ ]:
df_all = pd.merge(pred_and_init, df_ts_overlaps, left_on='Preferred Name', right_on='Preferred Name', how = 'left') #merge with overlap dataset

In [ ]:
# Count occurrences of 1, 0, and NaN in the 'cons_predictions' column
count_1 = df_all['predicted_activity'].eq(1).sum()  # Count where value is 1
count_0 = df_all['predicted_activity'].eq(0).sum()  # Count where value is 0
count_nan = df_all['predicted_activity'].isna().sum()  # Count NaN values
count_1_measured = df_all['NIS Hit2'].eq(1).sum() #NIS Hit2 is the column of the training set that conatins the classifications from in vitro results
count_0_measured = df_all['NIS Hit2'].eq(0).sum()

# Display the results
print(f"Count of 1 in 'predicted_activity': {count_1}")
print(f"Count of 0 in 'predicted_activity': {count_0}")
print(f"Count of NaN in 'predicted_activity': {count_nan}")
print(f"Count of 1 in 'NIS Hit2': {count_1_measured}")
print(f"Count of 0 in 'NIS Hit2': {count_0_measured}")

In [ ]:
#enter column names you want to keep in your output file
columns_to_keep = [
    'Preferred Name',
    'random_forest_Total_Prediction',
    'random_forest_Majority_Vote',
    'docking score',
    'normalized_docking_score_man',
    'normalized_random_forest_Total_Prediction_man',
    'normalized_sum_rf_man',
    'predicted_activity',
    'IUPAC Name',
    'Classification',
    'CASRN',
    'n. Positive',
    'n. Negative',
    'Consensus verdict',
    'Smiles',
    'NIS Hit2'
]

df_filtered = df_all[columns_to_keep].copy()

In [ ]:
df_filtered.to_csv('file_fin_relevant_columns.csv', index=False) #save final file

## 6. Applicabilty domain

In [ ]:
SDF_fin = pd.read_csv('file_standardized.csv') #your standardized data from 1. Data upload and standardization
merged_df = pd.read_csv('consensus_pred.csv') #your predictions table from 5. Consensus scoring (ML + docking)
df_training_set = pd.read_csv('NIS_stand_CDDD_filtered.csv') #training set

In [ ]:
#merge predictions with standardized data to get the standardized Smiles for each compound
merged_df = pd.merge(
    merged_df,
    SDF_fin[['Preferred Name', 'SMILES_stand']],
    on='Preferred Name',
    how='left'
)

In [ ]:
# Add test and training indicator columns
df_training_set["train"] = 1
df_training_set["test"] = 0

merged_df["train"] = 0 
merged_df["test"] = 1 

df_training_set = df_training_set.rename(columns={'SMILES_stand': 'smiles'})
merged_df = merged_df.rename(columns={'SMILES_stand': 'smiles'})

# Concatenate both DataFrames
combined_df = pd.concat([df_training_set, merged_df], ignore_index=True)


In [ ]:
cddd_columns = [f'CDDD fingerprint[{i}]' for i in range(1, 513)]

# Drop them from combined_df
filtered_combined_df = combined_df.drop(columns=cddd_columns, errors='ignore')

In [ ]:
from molcomplib import MolCompass
molcomp = MolCompass()
import pandas as pd
res = molcomp.process(combined_df)

In [ ]:
res_filtered = res[['x', 'y']].copy()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

# Subsets
test_data = combined_df[combined_df['test'] == 1]
train_data = combined_df[
    (combined_df['train'] == 1)]

# Define colors
train_colors = {
    (0): '#D8BFD8',   # Light Purple
    (1): '#5e3370'    # Dark Purple
}

# Define TP, TN, FN colors for ChEMBL
test_perf_colors = {
    'neg': '#FF8C00',     # Dark Orange
    'pos': '#B22222',     # Firebrick Red
    'NaN': '#B0B0B0'     # Grey
}

# Assign performance categories
def get_performance(row):
    if np.isnan(row['predicted_activity']):
        return 'NaN'
    elif row['predicted_activity'] == 1:
        return 'pos'
    elif row['predicted_activity'] == 0:
        return 'neg'
    else:
        return 'Other'  # Catch edge cases, e.g., predicted 1 but actual 0 (FP)

test_perf_labels = test_data.apply(get_performance, axis=1)
test_colors_list = [test_perf_colors.get(label, '#000000') for label in test_perf_labels]  # default black

# Assign colors for ToxCast
train_colors_list = [train_colors.get(row['NIS Hit2'], '#000000') for _, row in train_data.iterrows()]

# Create the plot
plt.figure(figsize=(10, 8))

# Plot ToxCast compounds
plt.scatter(train_data['x'], train_data['y'], c=train_colors_list, label='train', alpha=0.7)

# Plot ChEMBL compounds with performance-based coloring
plt.scatter(test_data['x'], test_data['y'], c=test_colors_list, label='test', alpha=0.7)

# Add labels and title
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.title('Compounds Scatter Plot')

# Custom legend
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='train (Inactive)', markerfacecolor='#D8BFD8', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='train (Non-cytotoxic, Active)', markerfacecolor='#5e3370', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='test (neg)', markerfacecolor='#FF8C00', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='test (pos)', markerfacecolor='#B22222', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='NaN', markerfacecolor='#B0B0B0', markersize=10)]

plt.legend(handles=legend_elements, loc='best')

#plt.grid(True)
plt.tight_layout()
plt.savefig("scatter.png", dpi=300)
plt.show()